# Cleaning RIPA Datasets
Based off of https://github.com/joshuagrossman/ripa/tree/main/src/ripa cleaning scripts
View documentation [here](https://github.com/laurenbchu/honors-thesis/blob/main/data/documentation/RIPA_2022_ReadMe.pdf)

In [23]:
import requests # To download files from the internet
import zipfile # To open zip files
import io # To treat downloaded data like a file in memory
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
def load_ripa_from_doj_zip(county):
    """
    Downloads DOJ RIPA Stop Data (2019–2023),
    loads only the Excel file that contains the county name,
    and returns a combined DataFrame.
    
    Example:
        df = load_ripa_from_doj_zip("Orange")
    """

    # DELETE LATER
    # url_dict = {2020: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2020.zip"}

    # DOJ public ZIP URLs for 2019–2023
    url_dict = {
        2019: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2019.zip",
        2020: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2020.zip",
        2021: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2023-03/RIPA-Stop-Data-2021.zip",
        2022: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2025-05/RIPA-Stop-Data-2022.zip",
        2023: "https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2025-05/RIPA-Stop-Data-2023.zip",
    }

    all_years = []

    for year, zip_url in url_dict.items():

        response = requests.get(zip_url)
        response.raise_for_status() # Checks whether website request succeeded

        z = zipfile.ZipFile(io.BytesIO(response.content)) # Opens the zip in memory

        # Find the Excel file containing the county name
        county_file = None # Creates placeholder value
        for file in z.namelist(): # For every file in the ZIP
            if county.lower() in file.lower() and file.lower().endswith(".xlsx"): # Check if file is the right county
                county_file = file
                break

        # If there's no file for that county
        if county_file is None:
            raise ValueError(f"No file found for {county} in {year}")

        # Open the file
        with z.open(county_file) as f:
            df_year = pd.read_excel(f, engine="openpyxl")

        # Make all columns lower-case strings
        df_year.columns = df_year.columns.str.lower()
        df_year["year"] = year # Add year column

        all_years.append(df_year)
        print(f"{year} loaded successfully.")

    final_df = pd.concat(all_years, ignore_index=True)

    return final_df

In [25]:
def add_derived_vars(df):
    """
    Adds the following derived variables:
      - race_ethnicity
      - gender
      - reason_for_contact
      - suspicion
      - action_any_search
      - search_basis_*
      - contraband_weapons
      - contraband_any
      - result_of_stop_arrest

    Returns a new DataFrame (copy) with added columns.
    """

    df = df.copy()

    # ------------------------------------------------------------
    # RACE/ETHNICITY
    # For each row, assign their corresponding race in a new column based on indicator columns
    # Ordered specifically so Hispanic/Latino comes before White, so if both races, labeled Hispanic/Latino
    # 1.2% of the dataset has more than one race listed, which hierarchical ordering handles
    # ------------------------------------------------------------
    
    df["race_ethnicity"] = np.select(
        [
            df["rae_hispanic_latino"] == 1,
            df["rae_black_african_american"] == 1,
            df["rae_asian"] == 1,
            df["rae_middle_eastern_south_asian"] == 1,
            df["rae_pacific_islander"] == 1,
            df["rae_native_american"] == 1,
            df["rae_white"] == 1,
        ],
        [
            "Hispanic",
            "Black",
            "Asian",
            "Middle Eastern/South Asian",
            "Pacific Islander",
            "Native American",
            "White",
        ],
        default="Other/Unknown"
    )

    # ------------------------------------------------------------
    # GENDER
    # ------------------------------------------------------------

    df["gender"] = np.select(
        [
            df["g_gender_nonconforming"] == 1,
            df["g_transgender_woman"] == 1,
            df["g_transgender_man"] == 1,
            df["g_female"] == 1,
            df["g_male"] == 1,
        ],
        [
            "Nonconforming",
            "Transgender Woman",
            "Transgender Man",
            "Female",
            "Male",
        ],
        default="Other/Unknown"
    )

    # ------------------------------------------------------------
    # REASON FOR CONTACT
    # ------------------------------------------------------------

    df["reason_for_contact"] = np.select(
        [
            (df["reason_for_stop"] == 1) & (df["rfs_traffic_violation_type"] == 1),
            (df["reason_for_stop"] == 1) & (df["rfs_traffic_violation_type"] == 2),
            (df["reason_for_stop"] == 1) & (df["rfs_traffic_violation_type"] == 3),
            df["reason_for_stop"] == 2,
            df["reason_for_stop"] == 3,
            df["reason_for_stop"] == 4,
            df["reason_for_stop"] == 6,
            df["reason_for_stop"].isin([5, 7, 8]),
        ],
        [
            "Moving violation",
            "Equipment violation",
            "Non-moving violation",
            "Suspect criminal activity",
            "Parole/probation stop",
            "Outstanding arrest",
            "Consensual search",
            "Truancy/School Related",
        ],
        default="Other/Unknown"
    )

    # ------------------------------------------------------------
    # SUSPICION CATEGORY
    # ------------------------------------------------------------

    df["suspicion"] = np.select(
        [
            df["rfs_rs_off_witness"] == 1,
            df["rfs_rs_match_suspect"] == 1,
            df["rfs_rs_witness_id"] == 1,
            df["rfs_rs_carry_sus_object"] == 1,
            df["rfs_rs_actions_indicative"] == 1,
            df["rfs_rs_suspect_look"] == 1,
            df["rfs_rs_drug_trans"] == 1,
            df["rfs_rs_violent_crime"] == 1,
            df["rfs_rs_reason_susp"] == 1,
        ],
        [
            "Officer witnessed commission of a crime",
            "Matched suspect description",
            "Witness or victim ID of suspect at the scene",
            "Carrying suspicious object",
            "Actions indicative of casing a victim or location",
            "Suspected of acting as a lookout",
            "Actions indicative of a drug transaction",
            "Actions indicative of engaging in a violent crime",
            "Other reasonable suspicion of a crime",
        ],
        default="None"
    )

    # ------------------------------------------------------------
    # ANY SEARCH OCCURRED
    # True if either person search or property search occurred
    # ------------------------------------------------------------

    df["action_any_search"] = (
        (df["ads_search_person"] == 1) |
        (df["ads_search_property"] == 1)
    )

    # ------------------------------------------------------------
    # SEARCH BASES
    # ------------------------------------------------------------

    df["search_basis_plain_view"] = df["bfs_visible_contraband"] == 1
    df["search_basis_plain_smell"] = df["bfs_odor_contraband"] == 1
    df["search_basis_consent"] = df["bfs_consent_given"] == 1
    df["search_basis_safety"] = df["bfs_officer_safety"] == 1
    df["search_basis_suspect_weapon"] = df["bfs_suspect_weapon"] == 1
    df["search_basis_evidence_of_crime"] = df["bfs_evidence"] == 1
    df["search_basis_school_policy"] = df["bfs_school_policy"] == 1
    df["search_basis_emergency"] = df["bfs_exigent_circum"] == 1
    df["search_basis_canine"] = df["bfs_canine_detect"] == 1
    df["search_basis_warrant"] = df["bfs_search_warrant"] == 1
    df["search_basis_probation"] = df["bfs_parole"] == 1
    df["search_basis_incident_to_arrest"] = df["bfs_incident"] == 1
    df["search_basis_vehicle_inventory"] = df["bfs_vehicle_invent"] == 1

    # ------------------------------------------------------------
    # CONTRABAND VARIABLES
    # ------------------------------------------------------------

    # Weapons found if either weapon OR firearm indicator equals 1
    df["contraband_weapons"] = (
        (df["ced_weapon"] == 1) |
        (df["ced_firearm"] == 1)
    )

    # Any contraband found if "no contraband" is NOT equal to 1
    df["contraband_any"] = df["ced_none_contraband"].fillna(0).ne(1)

    # ------------------------------------------------------------
    # ARREST RESULT
    # ------------------------------------------------------------

    df["result_of_stop_arrest"] = (
        (df["ros_custodial_without_warrant"] == 1) |
        (df["ros_custodial_warrant"] == 1)
    )

    # Return a clean copy (prevents pandas fragmentation warnings)
    return df.copy()

In [26]:
def add_search_basis_reasons(df):
    """
    Add two boolean flags indicating whether a stop includes:
      - any discretionary search basis
      - any non-discretionary search basis

    Note: a stop can be True for both if multiple bases are recorded.
    """

    out = df.copy()

    discretionary_cols = [
        "search_basis_plain_view",
        "search_basis_plain_smell",
        "search_basis_consent",
        "search_basis_safety",
        "search_basis_suspect_weapon",
        "search_basis_evidence_of_crime",
        "search_basis_emergency",
        "search_basis_canine",
    ]

    nondiscretionary_cols = [
        "search_basis_warrant",
        "search_basis_probation",
        "search_basis_incident_to_arrest",
        "search_basis_vehicle_inventory",
    ]

    disc_existing = [c for c in discretionary_cols if c in out.columns]
    nondisc_existing = [c for c in nondiscretionary_cols if c in out.columns]

    out["discretionary_search_basis"] = (
        out[disc_existing].fillna(False).any(axis=1) if disc_existing else False
    )

    out["nondiscretionary_search_basis"] = (
        out[nondisc_existing].fillna(False).any(axis=1) if nondisc_existing else False
    )

    return out

In [27]:
def add_multi_person_stop(df):
    """
    If multiple people were involved in one stop, they all have the same doj_record_id
    but with different person_number
    Add boolean multi-person stop indicator column
    """
    out = df.copy()

    stop_sizes = out.groupby("doj_record_id").size()
    out["multi_person_stop"] = out["doj_record_id"].map(stop_sizes) > 1

    return out

In [28]:
def add_offense_code_digits(df):
    """
    Extract numeric offense codes from text-based offense fields.

    Creates:
      - traffic_violation_cjis_offense_code
      - suspicion_cjis_offense_code

    Example:
        "VC 23152(a) - DUI"  →  "23152"
        "PC 245(a)(1)"       →  "245"
    """

    out = df.copy()

    # Convert to string and extract first sequence of digits
    out["traffic_violation_cjis_offense_code"] = (
        out["rfs_traffic_violation_code"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
    )

    # Reasonable suspicion offense code
    # Do the same thing
    out["suspicion_cjis_offense_code"] = (
        out["rfs_rs_code"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
    )

    return out

In [ ]:
# ONLY DO THIS ONCE because takes 77 minutes to run
# Load in the data
# orange = load_ripa_from_doj_zip("Orange")

# For my purposes, I saved the output locally
orange = pd.read_csv("../data/cleaned/orange_ripa_2019_2023.csv")

<positron-console-cell-29>:6: DtypeWarning: Columns (0: time_of_stop, 1: school_code, 2: school_name, 3: ros_warning_cds, 4: ros_citation_cds, 5: ros_in_field_cite_release_cds, 6: ros_custodial_wout_warrant_cds) have mixed types. Specify dtype option on import or set low_memory=False.


In [30]:
cleaned = add_derived_vars(orange)
cleaned = add_search_basis_reasons(cleaned)
cleaned = add_multi_person_stop(cleaned)
cleaned = add_offense_code_digits(cleaned)

cleaned.head()

,doj_record_id,person_number,agency_ori,agency_name,time_of_stop,date_of_stop,stop_duration,closest_city,school_code,school_name,stop_student,k12_school_grounds,rae_full,rae_asian,rae_black_african_american,rae_hispanic_latino,rae_middle_eastern_south_asian,rae_native_american,rae_pacific_islander,rae_white,rae_multiracial,g_full,g_male,g_female,g_transgender_man,g_transgender_woman,g_gender_nonconforming,g_multigender,lgbt,age,age_group,limited_english_fluency,pd_full,pd_deafness_hearing,pd_speech_impair,pd_blind,pd_mental_health,pd_devel_disab,pd_hyperactivity_disability,pd_other,pd_none_disability,pd_disab_multi,reason_for_stop,rfs_traffic_violation_type,rfs_traffic_violation_code,rfs_rs_code,rfs_rs_off_witness,rfs_rs_match_suspect,rfs_rs_witness_id,rfs_rs_carry_sus_object,rfs_rs_actions_indicative,rfs_rs_suspect_look,rfs_rs_drug_trans,rfs_rs_violent_crime,rfs_rs_reason_susp,rfs_ec_discipline_code,rfs_ec_discipline,call_for_service,ads_removed_vehicle_order,ads_removed_vehicle_phycontact,ads_sobriety_test,ads_curb_detent,ads_handcuffed,ads_patcar_detent,ads_canine_search,ads_firearm_point,ads_firearm_discharge,ads_elect_device,ads_impact_discharge,ads_canine_bite,ads_baton,ads_chem_spray,ads_other_contact,ads_photo,ads_asked_search_per,ads_search_person,ads_asked_search_prop,ads_search_property,ads_prop_seize,ads_vehicle_impound,ads_written_statement,ads_no_actions,ads_search_pers_consen,ads_search_prop_consen,bfs_consent_given,bfs_officer_safety,bfs_search_warrant,bfs_parole,bfs_suspect_weapon,bfs_visible_contraband,bfs_odor_contraband,bfs_canine_detect,bfs_evidence,bfs_incident,bfs_exigent_circum,bfs_vehicle_invent,bfs_school_policy,ced_none_contraband,ced_firearm,ced_ammunition,ced_weapon,ced_drugs,ced_alcohol,ced_money,ced_drug_paraphernalia,ced_stolen_prop,ced_elect_device,ced_other_contraband,bps_safekeeping,bps_contraband,bps_evidence,bps_impound_vehicle,bps_abandon_prop,bps_violate_school,tps_firearm,tps_ammunition,tps_weapon,tps_drugs,tps_alcohol,tps_money,tps_drug_paraphernalia,tps_stolen_prop,tps_cellphone,tps_vehicle,tps_contraband,ros_no_action,ros_warning,ros_citation,ros_in_field_cite_release,ros_custodial_warrant,ros_custodial_without_warrant,ros_field_interview_card,ros_noncriminal_transport,ros_contact_legal_guardian,ros_psych_hold,ros_us_homeland,ros_referral_school_admin,ros_referral_school_counselor,ros_warning_cds,ros_citation_cds,ros_in_field_cite_release_cds,ros_custodial_wout_warrant_cds,year,pd_multi,race_ethnicity,gender,reason_for_contact,suspicion,action_any_search,search_basis_plain_view,search_basis_plain_smell,search_basis_consent,search_basis_safety,search_basis_suspect_weapon,search_basis_evidence_of_crime,search_basis_school_policy,search_basis_emergency,search_basis_canine,search_basis_warrant,search_basis_probation,search_basis_incident_to_arrest,search_basis_vehicle_inventory,contraband_weapons,contraband_any,result_of_stop_arrest,discretionary_search_basis,nondiscretionary_search_basis,multi_person_stop,traffic_violation_cjis_offense_code,suspicion_cjis_offense_code
0,W300020069A6O99XAGK5,1,CA0300000,ORANGE CO SO,1117,2019-01-26,10,MISSION VIEJO,NaN,NaN,0,0,1,1,0,0,0,0,0,0,0,2,0,1,0,0,0,0,0,55,8,0,0,0,0,0,0,0,0,0,1,NaN,1.0,1.0,54106.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,0,0,0,0,0,0,0,0,NaN,54106,NaN,NaN,2019,NaN,Asian,Female,Moving violation,None,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,54106,NaN
1,W300020069D4XYP7190K,1,CA0300000,ORANGE CO SO,2030,2019-01-26,30,RANCHO SANTA MARGARITA,NaN,NaN,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,25,5,0,0,0,0,0,0,0,0,0,1,NaN,1.0,2.0,54109.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,

In [31]:
# Save the cleaned file to use in the analysis notebook
cleaned.to_csv("../data/cleaned/cleaned_ripa_orange_2019_2023.csv", index=False)